библиотеки

In [1]:
import pandas as pd

In [2]:
import os
os.getcwd()

'/home/alex/s21_projects/DSB7_Pandas.ID_1577650-1/src/ex01'

1) views

In [3]:
views = pd.read_csv(
    "../data/feed-views.log",
    sep=r"\s{2,}|\t",
    header=None,
    names=["datetime", "user"],
    engine="python",
    )


In [4]:
print(views.dtypes)

datetime    str
user        str
dtype: object


In [5]:
views["datetime"] = pd.to_datetime(views["datetime"]).astype("datetime64[ns]")

In [6]:
print(views.dtypes)

datetime    datetime64[ns]
user                   str
dtype: object


In [7]:
views.info()

<class 'pandas.DataFrame'>
RangeIndex: 1076 entries, 0 to 1075
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  1076 non-null   datetime64[ns]
 1   user      1076 non-null   str           
dtypes: datetime64[ns](1), str(1)
memory usage: 16.9 KB


2) daytime

In [8]:
views["year"] = views["datetime"].dt.year
views["month"] = views["datetime"].dt.month
views["day"] = views["datetime"].dt.day
views["hour"] = views["datetime"].dt.hour
views["minute"] = views["datetime"].dt.minute
views["second"] = views["datetime"].dt.second

print(views.head())

                    datetime   user  year  month  day  hour  minute  second
0 2020-04-17 12:01:08.463179  artem  2020      4   17    12       1       8
1 2020-04-17 12:01:23.743946  artem  2020      4   17    12       1      23
2 2020-04-17 12:27:30.646665  artem  2020      4   17    12      27      30
3 2020-04-17 12:35:44.884757  artem  2020      4   17    12      35      44
4 2020-04-17 12:35:52.735016  artem  2020      4   17    12      35      52


In [9]:
bins = [-1,3,6,10,16,19,23]
labels = ["night", "early morning", "morning", "afternoon", "early evening", "evening"]
views["daytime"] = pd.cut(views["hour"], bins=bins, labels=labels)
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)
views = views.set_index("user")
print(views.head())

                        datetime  year  month  day  hour  minute  second  \
user                                                                       
artem 2020-04-17 12:01:08.463179  2020      4   17    12       1       8   
artem 2020-04-17 12:01:23.743946  2020      4   17    12       1      23   
artem 2020-04-17 12:27:30.646665  2020      4   17    12      27      30   
artem 2020-04-17 12:35:44.884757  2020      4   17    12      35      44   
artem 2020-04-17 12:35:52.735016  2020      4   17    12      35      52   

         daytime  
user              
artem  afternoon  
artem  afternoon  
artem  afternoon  
artem  afternoon  
artem  afternoon  


In [10]:
views.hour.mode()

0    22
Name: hour, dtype: int32

3) count

In [11]:
views.count()

datetime    1076
year        1076
month       1076
day         1076
hour        1076
minute      1076
second      1076
daytime     1076
dtype: int64

In [12]:
views["daytime"].value_counts()

daytime
evening          509
afternoon        252
early evening    145
night            129
morning           36
early morning      5
Name: count, dtype: int64

4) sort

In [13]:
views = views.sort_values(by=["hour", "minute", "second"])
print(views.tail(10))

                            datetime  year  month  day  hour  minute  second  \
user                                                                           
artem     2020-04-18 23:40:32.666884  2020      4   18    23      40      32   
artem     2020-04-26 23:41:18.281836  2020      4   26    23      41      18   
valentina 2020-05-20 23:43:07.300734  2020      5   20    23      43       7   
artem     2020-04-22 23:46:37.937105  2020      4   22    23      46      37   
artem     2020-04-29 23:48:14.208828  2020      4   29    23      48      14   
artem     2020-05-21 23:49:22.386789  2020      5   21    23      49      22   
anatoliy  2020-05-09 23:53:55.599821  2020      5    9    23      53      55   
pavel     2020-05-09 23:54:54.260791  2020      5    9    23      54      54   
valentina 2020-05-14 23:58:56.754866  2020      5   14    23      58      56   
alexander 2020-05-14 23:59:38.758438  2020      5   14    23      59      38   

           daytime  
user              

5) min max

In [14]:
#максимальынй час ночью
max_hour = views.groupby("daytime")["hour"].max()["night"]
print(max_hour)

3


In [15]:
#минимальный час утром
min_hour = views.groupby("daytime")["hour"].min()["morning"]
print(min_hour)

8


In [16]:
max_hour_views = views[views["hour"] == max_hour]
print(max_hour_views.head(1))

                             datetime  year  month  day  hour  minute  second  \
user                                                                            
konstantin 2020-04-19 03:23:35.471598  2020      4   19     3      23      35   

           daytime  
user                
konstantin   night  


In [17]:
min_hour_views = views[views["hour"] == min_hour]
print(min_hour_views.head(1))

                            datetime  year  month  day  hour  minute  second  \
user                                                                           
alexander 2020-05-15 08:16:03.918402  2020      5   15     8      16       3   

           daytime  
user                
alexander  morning  


6) three earliest and latest hours of the day

In [18]:
views.nsmallest(3, "hour")["hour"]

user
valentina    0
valentina    0
pavel        0
Name: hour, dtype: int32

In [19]:
views.nlargest(3, "hour")["hour"]

user
ekaterina    23
ekaterina    23
ekaterina    23
Name: hour, dtype: int32

7) describe найти наиболее активный час

In [20]:
hour_stats = views["hour"].describe()
print(f"Наиболее активны с {hour_stats["25%"]} до {hour_stats["75%"]}" )
iqr = hour_stats["75%"] - hour_stats["25%"]
print(f"Квартиль = {iqr}")

Наиболее активны с 13.0 до 22.0
Квартиль = 9.0
